In [0]:
import uuid
import json
import hashlib
from datetime import datetime, timezone

# ============================================================
# Silver framework: writes silver tables (the only code that writes in silver).
#   reference tables: rebuilt from their origin on every run (INSERT OVERWRITE)
#   entities: the framework decides the scope of each mapping on every run:
#     incremental          bronze files newer than the last successful batch of the SAME mapping version
#     full_first_load      the mapping was never loaded
#     full_version_change  the mapping or its entity contract changed since the last successful batch
#     full_manual          mode='initial'
#   -> batches of ~silver_batch_files files -> parse -> quality
#   -> MERGE: insert new rows, update ONLY rows whose content changed, delete rows that disappeared
#      from the reprocessed files -> quarantine replaced for those files -> run log + quality log
#   Full scopes that finish without errors also remove rows of files that no longer exist in bronze.
# processed_at = when the row last changed (identical rows are not rewritten).
# A batch with status 'fail' writes nothing and stops that mapping; its files stay pending (retried next run).
# Tables are created from the contracts with informational PK / FK constraints (Unity Catalog).
# Requires: common/_utils_config, common/_utils_reference_data, silver/utils/_utils_silver_contracts,
#           silver/utils/_utils_silver_mappings, silver/utils/_utils_silver_parser,
#           silver/utils/_utils_silver_quality
# Cells: 1 names/DDL/versions · 2 logs and reference tables · 3 entity mappings · 4 run and reports
# ============================================================

quarantine_suffix = "_quarantine"
quarantine_extra_columns = [("failed_rules", "array<string>"), ("raw_values", "map<string,string>"),
                            ("quarantined_at", "timestamp")]
success_statuses = ("ok", "warning", "empty")          # batches that move the incremental watermark
status_severity = {"no_new_data": 0, "ok": 0, "empty": 0, "warning": 1, "invalid": 2, "fail": 2, "error": 3}
merge_ignored_columns = {"bronze_run_id", "silver_run_id", "processed_at"}   # not compared to detect changes

run_log_columns = [
    ("run_id", "string"), ("batch_number", "int"), ("mapping", "string"), ("entity", "string"),
    ("source_name", "string"), ("mode", "string"), ("scope", "string"), ("mapping_version", "string"),
    ("status", "string"), ("bronze_files", "int"), ("rows", "bigint"), ("valid_rows", "bigint"),
    ("quarantined_rows", "bigint"), ("rows_inserted", "bigint"), ("rows_updated", "bigint"),
    ("rows_deleted", "bigint"), ("max_bronze_ingested_at", "timestamp"), ("started_at", "timestamp"),
    ("finished_at", "timestamp"), ("error_message", "string"),
]
quality_log_columns = [
    ("run_id", "string"), ("batch_number", "int"), ("mapping", "string"), ("entity", "string"),
    ("rule", "string"), ("column_name", "string"), ("action", "string"), ("evaluated", "bigint"),
    ("failed", "bigint"), ("logged_at", "timestamp"),
]


def silver_run_log_table():
    return f"{catalog}.{silver_schema}.silver_run_log"


def silver_quality_log_table():
    return f"{catalog}.{silver_schema}.silver_quality_log"


def quarantine_table_name(entity):
    return silver_table_full_name(f"{entity}{quarantine_suffix}")


def quarantine_summary_view():
    return f"{catalog}.{silver_schema}.silver_quarantine_summary"


def now_utc():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def mapping_version(mapping, contract):
    """md5 of the mapping and its entity contract: any approved change produces a new version."""
    content = json.dumps({"mapping": mapping, "contract": contract}, sort_keys=True, default=str)
    return hashlib.md5(content.encode("utf-8")).hexdigest()


def ddl_columns(columns, not_null=(), comments=None):
    comments = comments or {}
    parts = []
    for name, column_type in columns:
        part = f"`{name}` {column_type.upper()}"
        if name in not_null:
            part += " NOT NULL"
        if comments.get(name):
            part += f" COMMENT {sql_string(comments[name])}"
        parts.append(part)
    return ",\n  ".join(parts)


def silver_table_ddl(table, contract):
    pk = table["primary_key"]["name"]
    comments = {c["name"]: c["description"] for c in contract["columns"]}
    constraints = [f"CONSTRAINT pk_{table['name']} PRIMARY KEY (`{pk}`)"]
    for fk in table["foreign_keys"]:
        if fk["references_table"] == table["name"]:
            continue                                   # self-reference (data_source): documented, not declared
        constraints.append(f"CONSTRAINT fk_{table['name']}_{fk['references_table']} FOREIGN KEY (`{fk['name']}`) "
                           f"REFERENCES {silver_table_full_name(fk['references_table'])} (`{fk['references_column']}`)")
    body = ddl_columns(table["columns"], not_null={pk}, comments=comments) + ",\n  " + ",\n  ".join(constraints)
    return (f"CREATE TABLE IF NOT EXISTS {silver_table_full_name(table['name'])} (\n  {body}\n) "
            f"COMMENT {sql_string(contract['description'])}")


def add_missing_columns(full_name, columns):
    """Contract evolution: columns added to a contract (or to a log) are added to the existing table."""
    existing = {field.name.lower() for field in spark.table(full_name).schema.fields}
    missing = [(name, column_type) for name, column_type in columns if name.lower() not in existing]
    if missing:
        spark.sql(f"ALTER TABLE {full_name} ADD COLUMNS ({ddl_columns(missing)})")
    return [name for name, _ in missing]


def ensure_silver_tables(contract):
    """Creates (or evolves) the table of a contract and, for entities, its quarantine table."""
    table = build_silver_tables(contract)[0]
    full_name = silver_table_full_name(table["name"])
    spark.sql(silver_table_ddl(table, contract))
    add_missing_columns(full_name, table["columns"])
    if table["role"] == "entity":
        quarantine = quarantine_table_name(contract["entity"])
        spark.sql(f"CREATE TABLE IF NOT EXISTS {quarantine} (\n  "
                  f"{ddl_columns(table['columns'] + quarantine_extra_columns)}\n)")
        add_missing_columns(quarantine, table["columns"] + quarantine_extra_columns)
    return table


def create_log_tables():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {silver_run_log_table()} (\n  {ddl_columns(run_log_columns)}\n)")
    add_missing_columns(silver_run_log_table(), run_log_columns)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {silver_quality_log_table()} (\n  {ddl_columns(quality_log_columns)}\n)")
    add_missing_columns(silver_quality_log_table(), quality_log_columns)

In [0]:
# ------------------------------------------------------------
# Logs and reference tables
# ------------------------------------------------------------
def log_run(rows):
    schema = ", ".join(f"`{name}` {column_type}" for name, column_type in run_log_columns)
    data = [tuple(row.get(name) for name, _ in run_log_columns) for row in rows]
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable(silver_run_log_table())


def log_quality(run_id, batch_number, mapping_name, entity, metrics):
    if not metrics:
        return
    schema = ", ".join(f"`{name}` {column_type}" for name, column_type in quality_log_columns)
    logged_at = now_utc()
    data = [(run_id, batch_number, mapping_name, entity, m["rule"], m["column"], m["action"],
             int(m["evaluated"]), int(m["failed"]), logged_at) for m in metrics]
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable(silver_quality_log_table())


def new_summary(name, entity, scope=None):
    return {"mapping": name, "entity": entity, "scope": scope, "status": "ok", "batches": 0, "files": 0,
            "rows": 0, "valid_rows": 0, "quarantined_rows": 0, "rows_inserted": 0, "rows_updated": 0,
            "rows_deleted": 0, "error": None}


def load_reference_table(contract, contracts, run_id):
    """Rebuilds a reference table from its origin (csv or source specs)."""
    entity = contract["entity"]
    summary = new_summary(f"reference__{entity}", entity, scope="full_reference")
    log = {"run_id": run_id, "batch_number": 1, "mapping": summary["mapping"], "entity": entity,
           "source_name": table_data_source(contract), "mode": "full", "scope": "full_reference",
           "bronze_files": 0, "started_at": now_utc()}
    try:
        ensure_silver_tables(contract)
        df = build_reference_df(contract, contracts, run_id)
        view = f"__silver_reference_{entity}"
        df.createOrReplaceTempView(view)
        spark.sql(f"INSERT OVERWRITE TABLE {silver_table_full_name(entity)} SELECT * FROM {view}")
        rows = spark.table(silver_table_full_name(entity)).count()
        log.update(status="ok", rows=rows, valid_rows=rows, quarantined_rows=0, rows_inserted=rows)
        summary.update(batches=1, rows=rows, valid_rows=rows, rows_inserted=rows)
    except Exception as e:
        log.update(status="error", error_message=f"{type(e).__name__}: {str(e)[:900]}")
        summary.update(status="error", error=log["error_message"])
    log["finished_at"] = now_utc()
    log_run([log])
    return summary

In [0]:
# ------------------------------------------------------------
# Entity mappings: scope -> pending bronze files -> batches -> parse -> quality -> MERGE + quarantine + logs
# ------------------------------------------------------------
def last_success_state(mapping_name):
    """(found, version, watermark) of the last successful batch of a mapping.
    The watermark only considers successful batches of that same version."""
    if not spark.catalog.tableExists(silver_run_log_table()):
        return False, None, None
    successes = (spark.table(silver_run_log_table())
                 .filter((F.col("mapping") == mapping_name) & F.col("status").isin(*success_statuses)))
    latest = successes.orderBy(F.desc("finished_at")).select("mapping_version").first()
    if latest is None:
        return False, None, None
    version = latest["mapping_version"]
    same_version = F.col("mapping_version") == version if version is not None else F.col("mapping_version").isNull()
    watermark = successes.filter(same_version).agg(F.max("max_bronze_ingested_at")).first()[0]
    return True, version, watermark


def decide_scope(mode, version, found, last_version):
    if mode == "initial":
        return "full_manual"
    if not found:
        return "full_first_load"
    if last_version != version:
        return "full_version_change"
    return "incremental"


def pending_bronze_files(mapping, watermark):
    """Bronze files (with their ingested_at) newer than the watermark (all if None), oldest first."""
    files = (spark.table(bronze_table_for_mapping(mapping))
             .groupBy("source_file")
             .agg(F.max("ingested_at").alias("ingested_at"), F.min("chunk_start").alias("chunk_start")))
    if watermark is not None:
        files = files.filter(F.col("ingested_at") > F.lit(watermark))
    return [r.asDict() for r in files.orderBy("ingested_at", "chunk_start").collect()]


def make_batches(files, batch_files):
    """Groups of about batch_files files. Files with the same ingested_at always stay in the same batch:
    the watermark is a timestamp, so a batch boundary must never split equal timestamps."""
    batches, current = [], []
    for i, file in enumerate(files):
        current.append(file)
        next_has_same_time = i + 1 < len(files) and files[i + 1]["ingested_at"] == file["ingested_at"]
        if len(current) >= batch_files and not next_has_same_time:
            batches.append(current)
            current = []
    if current:
        batches.append(current)
    return batches


def last_operation_metrics(full_name):
    metrics = spark.sql(f"DESCRIBE HISTORY {full_name} LIMIT 1").select("operationMetrics").first()[0] or {}
    return {"rows_inserted": int(metrics.get("numTargetRowsInserted", 0)),
            "rows_updated": int(metrics.get("numTargetRowsUpdated", 0)),
            "rows_deleted": int(metrics.get("numTargetRowsDeleted", 0))}


def write_entity_batch(mapping, contract, quality, files):
    """MERGE: insert new rows, update only rows whose content changed, delete rows that disappeared
    from these files. Then replace the quarantine of these files."""
    table = build_silver_tables(contract)[0]
    entity = contract["entity"]
    full_name = silver_table_full_name(entity)
    pk = table["primary_key"]["name"]
    source_name = sql_string(mapping["source"]["source_name"])
    file_list = ", ".join(sql_string(f["source_file"]) for f in files)
    compared = [name for name, _ in table["columns"] if name != pk and name not in merge_ignored_columns]
    changed = " OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in compared)

    view = f"__silver_batch_{entity}"
    quality["valid"].createOrReplaceTempView(view)
    spark.sql(f"""
        MERGE INTO {full_name} AS t
        USING {view} AS s
        ON t.`{pk}` = s.`{pk}`
        WHEN MATCHED AND ({changed}) THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        WHEN NOT MATCHED BY SOURCE AND t.source_name = {source_name} AND t.source_file IN ({file_list}) THEN DELETE
    """)
    metrics = last_operation_metrics(full_name)

    quarantine = quarantine_table_name(entity)
    spark.sql(f"DELETE FROM {quarantine} WHERE source_name = {source_name} AND source_file IN ({file_list})")
    if quality["quarantined_rows"]:
        columns = [name for name, _ in table["columns"] + quarantine_extra_columns]
        quality["quarantine"].select(*columns).write.mode("append").saveAsTable(quarantine)
    return metrics


def remove_unprocessed_files(mapping, contract, processed_files):
    """After a full scope: removes rows of this data source whose bronze file was not reprocessed."""
    if not processed_files:
        return 0
    source_name = sql_string(mapping["source"]["source_name"])
    file_list = ", ".join(sql_string(f) for f in processed_files)
    full_name = silver_table_full_name(contract["entity"])
    spark.sql(f"DELETE FROM {full_name} WHERE source_name = {source_name} AND source_file NOT IN ({file_list})")
    deleted = int((spark.sql(f"DESCRIBE HISTORY {full_name} LIMIT 1").select("operationMetrics").first()[0]
                   or {}).get("numDeletedRows", 0))
    spark.sql(f"DELETE FROM {quarantine_table_name(contract['entity'])} "
              f"WHERE source_name = {source_name} AND source_file NOT IN ({file_list})")
    return deleted


def run_mapping(mapping, contracts, run_id, mode="auto", file_limit=None, batch_files=None):
    """Runs one mapping; the scope is decided from its version. file_limit (tests only): keep only the
    most recent pending files."""
    use_utc_session()
    contract = contracts[mapping["entity"]]
    name, entity = mapping["mapping"], contract["entity"]
    version = mapping_version(mapping, contract)
    summary = new_summary(name, entity)
    base_log = {"run_id": run_id, "mapping": name, "entity": entity,
                "source_name": mapping["source"]["source_name"], "mode": mode, "mapping_version": version}

    try:
        ensure_silver_tables(contract)
        found, last_version, watermark = last_success_state(name)
        scope = decide_scope(mode, version, found, last_version)
        files = pending_bronze_files(mapping, watermark if scope == "incremental" else None)
        if file_limit:
            files = files[-file_limit:]
    except Exception as e:
        error = f"{type(e).__name__}: {str(e)[:900]}"
        log_run([{**base_log, "batch_number": 0, "status": "error", "error_message": error,
                  "started_at": now_utc(), "finished_at": now_utc()}])
        summary.update(status="error", error=error)
        return summary

    summary["scope"] = scope
    base_log["scope"] = scope
    if not files:
        log_run([{**base_log, "batch_number": 0, "status": "no_new_data", "bronze_files": 0,
                  "started_at": now_utc(), "finished_at": now_utc()}])
        summary["status"] = "no_new_data"
        print(f"  {name}: {scope} · no new bronze data")
        return summary

    statuses, processed_files = [], []
    batches = make_batches(files, batch_files or silver_batch_files)
    print(f"  {name}: {scope} · {len(files)} bronze file(s) in {len(batches)} batch(es)")
    for number, batch in enumerate(batches, start=1):
        log = {**base_log, "batch_number": number, "bronze_files": len(batch), "started_at": now_utc(),
               "max_bronze_ingested_at": max(f["ingested_at"] for f in batch)}
        quality = None
        try:
            paths = [f["source_file"] for f in batch]
            bronze_df = spark.table(bronze_table_for_mapping(mapping)).filter(F.col("source_file").isin(paths))
            parsed = parse_mapping(mapping, contracts, bronze_df, run_id)
            quality = evaluate_quality(parsed, mapping, contracts)
            log.update(status=quality["status"], rows=quality["rows"], valid_rows=quality["valid_rows"],
                       quarantined_rows=quality["quarantined_rows"])
            if quality["status"] == "fail":
                log["error_message"] = (f"quarantine ratio {quality['quarantine_ratio']:.1%} above "
                                        f"{max_quarantine_ratio:.0%}: nothing written")
            elif quality["status"] != "empty":
                log.update(write_entity_batch(mapping, contract, quality, batch))
                processed_files += paths
        except Exception as e:
            log.update(status="error", error_message=f"{type(e).__name__}: {str(e)[:900]}")
        log["finished_at"] = now_utc()
        log_run([log])
        if quality:
            log_quality(run_id, number, name, entity, quality["metrics"])

        statuses.append(log["status"])
        for key in ("rows", "valid_rows", "quarantined_rows", "rows_inserted", "rows_updated", "rows_deleted"):
            summary[key] += log.get(key) or 0
        summary["batches"] += 1
        summary["files"] += len(batch)
        print(f"    batch {number}/{len(batches)}: {log['status']} · {len(batch)} file(s) · "
              f"{log.get('valid_rows') or 0} valid · {log.get('quarantined_rows') or 0} quarantined · "
              f"+{log.get('rows_inserted') or 0} ~{log.get('rows_updated') or 0} -{log.get('rows_deleted') or 0}"
              + (f" · {log['error_message'][:150]}" if log.get("error_message") else ""))
        if log["status"] in ("fail", "error"):
            summary["error"] = log.get("error_message")
            break

    if scope != "incremental" and statuses and all(s in success_statuses for s in statuses):
        summary["rows_deleted"] += remove_unprocessed_files(mapping, contract, processed_files)
    summary["status"] = max(statuses, key=lambda s: status_severity[s])
    return summary

In [0]:
# ------------------------------------------------------------
# Full silver run, quarantine summary and reports
# ------------------------------------------------------------
def refresh_quarantine_summary_view(contracts):
    """One row per entity, data source and failed rule (a row failing two rules counts in both)."""
    parts = []
    for contract in contracts.values():
        quarantine = quarantine_table_name(contract["entity"])
        if contract["table_type"] == "entity" and spark.catalog.tableExists(quarantine):
            parts.append(f"SELECT {sql_string(contract['entity'])} AS entity, source_name, rule, quarantined_at "
                         f"FROM (SELECT source_name, explode(failed_rules) AS rule, quarantined_at FROM {quarantine})")
    if parts:
        spark.sql(f"CREATE OR REPLACE VIEW {quarantine_summary_view()} AS "
                  f"SELECT entity, source_name, rule, count(*) AS failed_rows, "
                  f"min(quarantined_at) AS first_quarantined_at, max(quarantined_at) AS last_quarantined_at "
                  f"FROM ({' UNION ALL '.join(parts)}) GROUP BY entity, source_name, rule")


def show_quarantine(entity, rule=None, limit=20):
    df = spark.table(quarantine_table_name(entity))
    if rule:
        df = df.filter(F.array_contains("failed_rules", rule))
    display(df.orderBy(F.desc("quarantined_at")).limit(limit))


def print_run_summary(run_id, summaries):
    print(f"\n===== SILVER RUN {run_id} =====")
    print(f"   {'mapping':<48} {'scope':<20} {'status':<12} {'files':>5} {'valid':>9} {'quar.':>6} "
          f"{'ins':>8} {'upd':>8} {'del':>6}")
    for s in summaries:
        icon = {"ok": "✅", "no_new_data": "· ", "warning": "⚠️", "empty": "∅ "}.get(s["status"], "⛔")
        print(f"{icon} {s['mapping']:<48} {str(s['scope'] or '-'):<20} {s['status']:<12} {s['files']:>5} "
              f"{s['valid_rows']:>9} {s['quarantined_rows']:>6} {s['rows_inserted']:>8} {s['rows_updated']:>8} "
              f"{s['rows_deleted']:>6}")
        if s["error"]:
            print(f"     ↳ {s['error'][:300]}")


def run_silver(mode="auto", only=None, file_limit=None, batch_files=None):
    """mode: auto (the framework decides the scope of each mapping) or initial (reprocess all bronze).
    only: list of mapping names. Raises at the end if any mapping failed (so a Job task fails and notifies)."""
    if mode not in ("auto", "initial"):
        raise ValueError("mode must be 'auto' or 'initial'")
    use_utc_session()
    run_id = str(uuid.uuid4())
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")
    create_log_tables()

    contracts = load_all_entity_contracts()
    contract_errors = validate_all_entity_contracts(contracts)
    if contract_errors:
        raise ValueError(f"Invalid entity contracts, silver run stopped: {contract_errors}")
    print(f"▶ silver run {run_id} · mode {mode} · schema {catalog}.{silver_schema}")

    summaries = []
    references = sorted((c for c in contracts.values() if c["table_type"] == "reference"),
                        key=lambda c: (c["entity"] != data_source_entity, c["entity"]))
    for contract in references:
        summaries.append(load_reference_table(contract, contracts, run_id))

    mappings = load_all_mappings()
    mapping_errors = validate_all_mappings(contracts, mappings)
    for name in sorted(mappings):
        if only and name not in only:
            continue
        if name in mapping_errors:
            entity = mappings[name].get("entity")
            error = f"invalid mapping: {mapping_errors[name]}"
            log_run([{"run_id": run_id, "batch_number": 0, "mapping": name, "entity": entity, "mode": mode,
                      "status": "invalid", "error_message": error[:1000],
                      "started_at": now_utc(), "finished_at": now_utc()}])
            summary = new_summary(name, entity)
            summary.update(status="invalid", error=error)
            summaries.append(summary)
            continue
        summaries.append(run_mapping(mappings[name], contracts, run_id, mode, file_limit, batch_files))

    refresh_quarantine_summary_view(contracts)
    print_run_summary(run_id, summaries)
    problems = [s["mapping"] for s in summaries if status_severity[s["status"]] >= 2]
    if problems:
        raise RuntimeError(f"Silver run {run_id}: {len(problems)} mapping(s) failed: {problems}")
    return {"run_id": run_id, "summaries": summaries}